# Sentio.io — model training (Colab)

Aspect-based sentiment analysis of code-mixed Hinglish food delivery reviews.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save.

Then run each cell from top to bottom (Shift + Enter). Total time is about 20–25 minutes.

| Step | What it does |
|---|---|
| 1 | Checks the GPU |
| 2 | Uploads `sentio_colab.zip` (data + scripts) |
| 3 | Connects Google Drive so the trained model is kept |
| 4 | Trains **MuRIL** (main model) |
| 5 | Trains mBERT, XLM-R, HingRoBERTa for the comparison table |
| 6 | Builds the comparison table |
| 7 | Shows mistakes for error analysis |
| 8 | Saves the best model to Drive and downloads the results |

The **test split is frozen**: every model picks its best epoch on the validation split and is scored on test exactly once.

In [ ]:
# 1. GPU check — should print a Tesla T4
!nvidia-smi -L
import torch, transformers
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| transformers", transformers.__version__)
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU, then run this cell again."

In [ ]:
# 2. Upload sentio_colab.zip (from your laptop: D:\\Coding\\Mini Project\\ML\\colab\\)
from google.colab import files
uploaded = files.upload()
!unzip -o -q sentio_colab.zip -d /content
%cd /content/sentio_colab
!ls -R | head -30
!cat splits_v1/split_report.txt

In [ ]:
# 3. Connect Google Drive (a pop-up asks you to allow access)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/Sentio'
import os; os.makedirs(DRIVE_DIR, exist_ok=True)
print("Models will be saved to", DRIVE_DIR)

## 4. Train MuRIL (main model)
One shared encoder, 8 heads: overall sentiment + one per aspect. About 3–5 minutes on a T4.

In [ ]:
!python train.py --model google/muril-base-cased --tag muril_v1

## 5. Comparison models (ablation)
Same data, same split, same settings — only the pretrained encoder changes.
This table is the strongest evidence that the results are your own work.

In [ ]:
!python train.py --model bert-base-multilingual-cased --tag mbert
!python train.py --model xlm-roberta-base --tag xlmr
!python train.py --model l3cube-pune/hing-roberta --tag hingroberta

In [ ]:
# 6. Comparison table (all scores on the same frozen test split)
!python compare_results.py

In [ ]:
# 7. Error analysis — MuRIL's mistakes on overall sentiment
import pandas as pd
pd.set_option('display.max_colwidth', 120)
p = pd.read_csv('results/muril_v1_test_predictions.csv')
err = p[p.overall_sentiment != p.pred_overall_sentiment]
print(f"{len(err)} of {len(p)} test reviews have the wrong overall sentiment")
print(pd.crosstab(p.overall_sentiment, p.pred_overall_sentiment, rownames=['gold'], colnames=['predicted']))
err[['id', 'review_text', 'overall_sentiment', 'pred_overall_sentiment']].head(25)

In [ ]:
# 8. Save the best model to Google Drive + download the results
# "Best" is chosen on the VALIDATION score, never on test.
import glob, json, shutil, torch
scores = {}
for f in glob.glob('results/*.json'):
    j = json.load(open(f))
    if 'best_val_score' in j:
        scores[j['tag']] = j['best_val_score']
best = max(scores, key=scores.get)
print("Validation scores:", scores)
print("Best model (by validation):", best)

for tag in sorted({best, 'muril_v1'}):
    src, dst = f'models/{tag}', f'{DRIVE_DIR}/models/{tag}'
    os.makedirs(dst, exist_ok=True)
    # store weights in float16: half the size, same predictions
    sd = torch.load(f'{src}/model.pt', map_location='cpu', weights_only=True)
    torch.save({k: v.half() if v.is_floating_point() else v for k, v in sd.items()}, f'{dst}/model.pt')
    for extra in glob.glob(f'{src}/*'):
        if not extra.endswith('model.pt'):
            shutil.copy(extra, dst)
    print("saved", dst, f"({os.path.getsize(dst + '/model.pt') / 1e6:.0f} MB)")

shutil.copytree('results', f'{DRIVE_DIR}/results', dirs_exist_ok=True)
shutil.copytree('logs', f'{DRIVE_DIR}/logs', dirs_exist_ok=True)
!zip -q -r sentio_results.zip results logs
files.download('sentio_results.zip')

### Done
- `sentio_results.zip` (small) → unzip into `D:\Coding\Mini Project\ML\` on your laptop
- Trained model(s) → Google Drive, `MyDrive/Sentio/models/`